In [1]:
#| default_exp restxl

In [5]:
from torch import inf

In [8]:
#| hide
import nbdev; nbdev.nbdev_export()

In [7]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [4]:
#| export
from rest.core import init_instance, generate
singleton, model_path = init_instance()

In [16]:
from transformers.file_utils import url_to_filename

In [17]:
url_to_filename("sberbank-ai/rugpt3xl")

'8ed537e3e1c420da44cc791bc35fb86a85cb7422435768fdf0a62409c51bb614'

In [19]:
from transformers import GPT2Tokenizer, PreTrainedModel, PretrainedConfig
tokenizer = GPT2Tokenizer.from_pretrained("sberbank-ai/rugpt3xl", local_files_only=True)


Distant resource does not have an ETag, we won't be able to reliably ensure reproducibility.


In [10]:
from transformers import GPT2Tokenizer, PreTrainedModel, PretrainedConfig
tokenizer = GPT2Tokenizer.from_pretrained("sberbank-ai/rugpt3xl", local_files_only=True)


Distant resource does not have an ETag, we won't be able to reliably ensure reproducibility.


In [20]:
tokenizer.save_pretrained('./tokenizer/rugpt3xl.tokenizer')

('./tokenizer/rugpt3xl.tokenizer/tokenizer_config.json',
 './tokenizer/rugpt3xl.tokenizer/special_tokens_map.json',
 './tokenizer/rugpt3xl.tokenizer/vocab.json',
 './tokenizer/rugpt3xl.tokenizer/merges.txt',
 './tokenizer/rugpt3xl.tokenizer/added_tokens.json')

In [9]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length,
    #local_files_only=True
)
tokenizer = model.tokenizer
model.cuda()
model.eval();

ModuleNotFoundError: No module named 'apex'

In [ ]:
sum(p.numel() for p in model.parameters())

1315737600

In [5]:
#| export
import deepspeed, torch
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.half,
                                 checkpoint=None,
                                 replace_method='auto',
                                 replace_with_kernel_inject=True)
model = ds_engine.module

NameError: name 'model' is not defined

In [7]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    return generate(model, tokenizer, seq_length, prompt, length, num_samples, allow_linebreak)

In [8]:
%%time
get_sample(' - ты кто? \n - ', 50, 4, False)

IndexError: map::at

In [11]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

 Дрозд? Гриф? Беркут? Гром?
Всех я видел зёрен незримых слитки.
Ты вот выпьешь литр молочной воды -
И будешь весь мой, весь в моей крови!

CPU times: user 49.4 s, sys: 1.97 s, total: 51.4 s
Wall time: 2.04 s
